In [ ]:
import os, numpy as np, pandas as pd
from tqdm.auto import tqdm

print(os.listdir('/kaggle/input/datasets/abhiroopsarkar'))     # confirm names first

ROOT = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
DFS  = '/kaggle/input/datasets/abhiroopsarkar/knee-shards-all/shards'
DNFS = '/kaggle/input/datasets/abhiroopsarkar/knee-shards-nfs'   # adjust if nested in train_nfs/
TMP  = '/kaggle/temp/xs'
LBL  = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
        'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
os.makedirs(TMP, exist_ok=True)
print(sorted(os.listdir(DNFS))[:3], len(os.listdir(DNFS)))

tr = pd.read_csv(f'{ROOT}/train.csv')
rl = pd.read_csv('/kaggle/input/datasets/abhiroopsarkar/knee-labels/report_labels.csv')
rl = rl.set_index('StudyInstanceUID').loc[tr.StudyInstanceUID].reset_index()
Y  = rl[LBL].values.astype(np.float32)
G  = tr[LBL].values.astype(np.float32)
gold = ~np.isnan(G).any(1)
print('gold:', gold.sum())

masks = []
for ci in tqdm(range(18)):
    a = np.load(f'{DFS}/shard_{ci:04d}.npz')
    b = np.load(f'{DNFS}/shard_{ci:04d}.npz')
    assert (a['uid'] == b['uid']).all(), f'uid mismatch shard {ci}'
    np.save(f'{TMP}/x_{ci:04d}.npy', np.concatenate([a['x'], b['x']], axis=1))
    masks.append(np.concatenate([a['m'], b['m']], axis=1))
    del a, b

MASK = np.concatenate(masks).astype(np.float32)
print('MASK', MASK.shape, 'coverage', MASK.mean(0).round(3))
assert len(MASK) == len(tr)

In [ ]:
import warnings, timm, torch, torch.nn as nn, torch.nn.functional as F, time
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore', category=UserWarning)

BACKBONE, KU, EPOCHS, BS, LR, P = 'efficientnet_b0', 16, 7, 4, 3e-4, 5
dev = 'cuda'
torch.backends.cudnn.benchmark = True

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train, s.mm = idx, train, {}
    def __len__(s): return len(s.idx)
    def __getitem__(s, j):
        i = s.idx[j]; ci = i // 256
        if ci not in s.mm: s.mm[ci] = np.load(f'{TMP}/x_{ci:04d}.npy', mmap_mode='r')
        sl = (np.sort(np.random.choice(24, KU, replace=False)) if s.train
              else np.linspace(0, 23, KU).round().astype(int))
        return (torch.from_numpy(np.array(s.mm[ci][i % 256][:, sl])),
                torch.from_numpy(MASK[i]), torch.from_numpy(Y[i]))

class Net(nn.Module):
    def __init__(s, pretrained=True):
        super().__init__()
        s.bb = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0, in_chans=1)
        D = s.bb.num_features
        s.att  = nn.ModuleList([nn.Linear(D, 1) for _ in range(P)])
        s.drop = nn.Dropout(0.3)
        s.head = nn.Linear(P * D + P, 12)
    def forward(s, x, m):
        B, Pp, K, H, W = x.shape
        f = s.bb(x.reshape(B * Pp * K, 1, H, W)).float().reshape(B, Pp, K, -1)
        e = []
        for p in range(P):
            w = torch.softmax(s.att[p](f[:, p]).squeeze(-1), 1)
            e.append((w.unsqueeze(-1) * f[:, p]).sum(1))
        e = torch.stack(e, 1) * m.unsqueeze(-1)
        return s.head(s.drop(torch.cat([e.flatten(1), m], 1)))

def prep(x): return (x.float() / 255 - 0.2) / 0.25
def aug(x):
    B = x.shape[0]
    x = x * (1 + 0.15 * torch.randn(B,1,1,1,1, device=x.device)) + 0.15 * torch.randn(B,1,1,1,1, device=x.device)
    dy, dx = np.random.randint(-10, 11, 2)
    return torch.roll(x, (int(dy), int(dx)), (3, 4))

def macro_auc(Pm, T):
    T = T >= 0.5
    return float(np.mean([roc_auc_score(T[:,k], Pm[:,k]) for k in range(12)
                          if 0 < T[:,k].sum() < len(T)]))

@torch.no_grad()
def predict(model, idx):
    model.eval(); out = []
    for x, m, _ in DataLoader(DS(idx, False), batch_size=6, num_workers=4):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(model(prep(x.to(dev)), m.to(dev)).float().sigmoid().cpu())
    return torch.cat(out).numpy()

rng = np.random.RandomState(0)
nong = np.where(~gold)[0]; rng.shuffle(nong)
nv = int(0.15 * len(nong)); val_idx, tr_idx = nong[:nv], nong[nv:]
gold_idx = np.where(gold)[0]
print('train', len(tr_idx), '| val', len(val_idx), '| gold', len(gold_idx), flush=True)

model = Net().to(dev).to(memory_format=torch.channels_last)
dl = DataLoader(DS(tr_idx, True), batch_size=BS, shuffle=True, num_workers=4,
                drop_last=True, pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS*len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')
best = 0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tl = 0
    for it, (x, m, y) in enumerate(dl):
        x, m, y = aug(prep(x.to(dev))), m.to(dev), y.to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            logit = model(x, m)
        loss = F.binary_cross_entropy_with_logits(logit.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
        if it % 400 == 0: print(f'ep{ep} it{it}/{len(dl)} loss {tl/(it+1):.4f} {time.time()-t0:.0f}s', flush=True)
    Pv, Pg = predict(model, val_idx), predict(model, gold_idx)
    pv, pg = macro_auc(Pv, Y[val_idx]), macro_auc(Pg, G[gold_idx])
    Tg = G[gold_idx] >= 0.5
    mm = roc_auc_score(Tg[:,2], Pg[:,2]); lm = roc_auc_score(Tg[:,3], Pg[:,3])
    print(f'=== ep{ep} loss {tl/len(dl):.4f} | val {pv:.3f} | GOLD {pg:.3f} | '
          f'MedMen {mm:.3f} | LatMen {lm:.3f} | {time.time()-t0:.0f}s', flush=True)
    sel = 0.5 * pv + 0.5 * pg
    if sel > best:
        best = sel; torch.save(model.state_dict(), '/kaggle/working/model_5p.pt')
        np.save('/kaggle/working/gold_5p.npy', Pg)     # for the paired bootstrap later